# 🏏 Midwicket Quickstart

**Welcome to Midwicket** — the Python library for agentic cricket analytics.

This notebook shows the two main APIs:
- **Win Probability** — live match win predictions
- **Win Probability Timeline** — over-by-over charts

> **No download required.** All examples run entirely in memory — zero waiting.

In [ ]:
# ── Step 1: Install Midwicket ──────────────────────────────────────────────
# This is the only step that touches the network. Typically takes ~20 seconds.
!pip install midwicket -q

In [ ]:
# ── Step 2: Verify install ─────────────────────────────────────────────────
import midwicket
print(f"✅ Midwicket {midwicket.__version__} loaded successfully")

---
## 🎯 Win Probability (Single Prediction)

Use `px.predict_win()` to get a live win probability estimate from any match state.
This is **instant** — it uses a lightweight model and requires no external data.

In [ ]:
import midwicket.express as px

# Match scenario: chasing 180, currently 120/5 after 15 overs at Wankhede
result = px.predict_win(
    venue="Wankhede Stadium",
    target=180,
    current_score=120,   # <-- runs scored so far
    wickets_down=5,
    overs_done=15.0,
)

print(f"🏏 Scenario: Chasing 180, currently {120}/{5} after {15} overs")
print(f"📊 Win Probability : {result['win_prob']:.1%}")
print(f"🔍 Confidence Score: {result['confidence']:.1%}")

---
## 📈 Win Probability Timeline (Over-by-Over)

Simulate how win probability shifts over the course of a chase — entirely in-memory,
no file downloads needed.

In [ ]:
import midwicket.express as px

# ── Synthetic match state: over-by-over snapshot ──────────────────────────
# Each tuple = (overs_done, current_score, wickets_down)
MATCH_STATES = [
    (0,  0,   0),
    (2,  18,  0),
    (4,  32,  1),
    (6,  55,  1),
    (8,  72,  2),
    (10, 95,  2),
    (12, 110, 3),
    (14, 118, 4),
    (16, 130, 5),
    (18, 152, 6),
    (20, 165, 8),
]

TARGET  = 180
VENUE   = "Wankhede Stadium"

# ── Compute win probability at each snapshot ───────────────────────────────
timeline = []
for overs, score, wkts in MATCH_STATES:
    r = px.predict_win(
        venue=VENUE,
        target=TARGET,
        current_score=score,
        wickets_down=wkts,
        overs_done=overs,
    )
    timeline.append((overs, score, wkts, r["win_prob"]))

# ── Pretty-print the timeline ──────────────────────────────────────────────
print(f"{'Over':>5}  {'Score':>8}  {'Win Prob':>10}")
print("-" * 30)
for overs, score, wkts, wp in timeline:
    bar = "█" * int(wp * 20)
    print(f"{overs:>5}  {score:>4}/{wkts:<3}  {wp:>8.1%}  {bar}")

---
## 📊 Plot the Timeline (Optional)

If `matplotlib` is available (it is in Colab by default), visualize the win probability arc.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    overs_list = [t[0] for t in timeline]
    wp_list    = [t[3] * 100 for t in timeline]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.fill_between(overs_list, wp_list, alpha=0.15, color="#4A90D9")
    ax.plot(overs_list, wp_list, marker="o", linewidth=2.5,
            color="#4A90D9", markerfacecolor="white", markersize=7)
    ax.axhline(50, color="gray", linestyle="--", linewidth=1, label="50% mark")
    ax.set_title(f"Win Probability Timeline — Chasing {TARGET} at {VENUE}",
                 fontsize=14, fontweight="bold")
    ax.set_xlabel("Overs Completed", fontsize=12)
    ax.set_ylabel("Win Probability (%)", fontsize=12)
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 100)
    ax.legend()
    plt.tight_layout()
    plt.show()
    print("✅ Chart rendered.")
except ImportError:
    print("Install matplotlib to see the chart: pip install matplotlib")

---
## 🚀 What's next?

| Feature | Code |
|---|---|
| Full historical dataset | `from midwicket.data.loader import DataLoader; DataLoader().download()` |
| Player stats | `px.get_player_stats("V Kohli")` |
| Head-to-head matchups | `px.get_matchup("V Kohli", "JJ Bumrah")` |
| Load all IPL matches | `px.quick_load()` |

📖 **Docs & Source:** [github.com/CodersAcademy006/Midwicket](https://github.com/CodersAcademy006/Midwicket)